# 01f - Random Search de Hiperparâmetros (fase 3)

Fase 3: afina os **hiperparâmetros do modelo** (learning rate, dropout, weight decay/L2) em cima da receita de augmentation vencedora (**jpeg+noise**, p_apply=0.6).

**Random search** — o espaço é **contínuo e multidimensional** (LR, dropout, WD), onde random search supera grid (Bergstra & Bengio 2012).

- **Augmentation fixa**: jpeg+noise (vencedor da fase 2)
- **Busca**: LR log-uniforme [1e-5, 1e-3], dropout [0.2, 0.7], weight_decay log-uniforme [1e-6, 1e-3] (+ chance de 0)
- **Regime**: 5%/10ep, métrica = AUC ArtiFact (cross-generator), 2 seeds por config

As classes de augmentation/dataset vêm de `aug_utils.py` (módulo importável) — isso permite **`num_workers>0` no Windows** (spawn pickla por referência ao módulo).

> ⚠️ Variância entre seeds (~0.04) é da ordem do efeito esperado dos HP. Defaults já são decentes → ganho provável modesto (+0.01–0.03). Cumpre o requisito do enunciado (ajuste de HP + regularização).

Baselines: raw=0.632 | augmentation jpeg+noise (HP default) = 0.687.

In [ ]:
import sys
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

# garante que aug_utils.py (ao lado do notebook) seja importável, inclusive pelos workers
sys.path.insert(0, str(Path.cwd()))
from aug_utils import RandomAugment, PathListDataset, clean_transform, train_transform

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

RAW_DIR       = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR  = DATA_ROOT / "raw" / "artifact_faces"
RESULTS_DIR   = PROJECT_ROOT / "artifacts" / "hp_search"
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH  = RESULTS_DIR / "hp_search_results.json"

IMAGE_SIZE      = 224
BATCH_SIZE      = 32
NUM_WORKERS     = 4           # workers OK: classes vêm de aug_utils.py (módulo importável)
NUM_EPOCHS      = 10
SAMPLE_FRACTION = 0.05
ARTIFACT_N_PER_CLASS = 2000
ARTIFACT_EVAL_SEED   = 42

# augmentation vencedora da fase 2 (fixa)
AUG_POOL   = ["jpeg", "noise"]
P_APPLY    = 0.6
AUG_RANGES = {"jpeg": (50, 90), "noise": (0.0, 0.05)}

# random search de HP
N_TRIALS = 20
SEEDS    = [42, 123]
# total = 20 * 2 = 40 runs

BASELINES = {"raw": 0.632, "jpeg+noise (HP default)": 0.687}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def sample_hp(trial):
    rng = random.Random(5000 + trial)
    lr = 10 ** rng.uniform(-5, -3)
    dropout = round(rng.uniform(0.2, 0.7), 2)
    wd = 0.0 if rng.random() < 0.25 else 10 ** rng.uniform(-6, -3)
    return {"lr": lr, "dropout": dropout, "weight_decay": wd}

print("Device:", DEVICE)
print(f"Augmentation fixa: {AUG_POOL} (p_apply={P_APPLY}) ranges={AUG_RANGES}")
print(f"num_workers={NUM_WORKERS}")
print(f"Trials: {N_TRIALS} x {len(SEEDS)} seeds = {N_TRIALS*len(SEEDS)} runs")

## 1. Transforms e dados (ArtiFact limpo, fixo e balanceado)

In [ ]:
AUGMENT  = RandomAugment(AUG_POOL, P_APPLY, AUG_RANGES)
TRAIN_TF = train_transform(IMAGE_SIZE, AUGMENT)
CLEAN_TF = clean_transform(IMAGE_SIZE)

def _list(folder):
    fs = []
    for e in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        fs += list(folder.glob(e))
    return sorted(fs)

# ArtiFact LIMPO, fixo e balanceado (mesmas imagens em todos os trials)
_rng = random.Random(ARTIFACT_EVAL_SEED)
_real = _rng.sample(_list(ARTIFACT_DIR / "real"), ARTIFACT_N_PER_CLASS)
_fake = _rng.sample(_list(ARTIFACT_DIR / "fake"), ARTIFACT_N_PER_CLASS)
ARTIFACT_ITEMS = [(p, 1) for p in _real] + [(p, 0) for p in _fake]
artifact_loader = DataLoader(PathListDataset(ARTIFACT_ITEMS, CLEAN_TF),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"ArtiFact eval (limpo): {len(_real)} real + {len(_fake)} fake")

def sample_subset(dataset, fraction, seed):
    n = int(len(dataset) * fraction)
    idx = random.Random(seed).sample(range(len(dataset)), n)
    return Subset(dataset, idx)

def build_train_valid(seed):
    train_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "train", transform=TRAIN_TF), SAMPLE_FRACTION, seed)
    valid_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "valid", transform=CLEAN_TF), SAMPLE_FRACTION, seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, valid_loader

print("Loaders definidos.")

## 2. Treino com HP configurável

In [ ]:
def build_model(dropout, seed=42):
    torch.manual_seed(seed)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_f = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, 2))
    return model.to(DEVICE)

@torch.no_grad()
def auc_on(model, loader):
    model.eval()
    yt, yp = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        p = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
        yp.extend(p.tolist()); yt.extend(lbls.tolist())
    return float(roc_auc_score(yt, yp))

def run_trial(hp, seed):
    train_loader, valid_loader = build_train_valid(seed)
    model = build_model(hp["dropout"], seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    for _ in range(NUM_EPOCHS):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
    val_auc      = auc_on(model, valid_loader)
    artifact_auc = auc_on(model, artifact_loader)
    del model; torch.cuda.empty_cache()
    return val_auc, artifact_auc

print("Funções de treino definidas.")

## 3. Random Search

Salva incrementalmente — retoma de onde parou.

In [ ]:
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    done = {(r["trial"], r["seed"]) for r in results}
    print(f"Retomando: {len(results)} runs já feitos.")
else:
    results = []
    done = set()

jobs = [(t, s) for t in range(N_TRIALS) for s in SEEDS if (t, s) not in done]
print(f"Runs restantes: {len(jobs)}")

for trial, seed in tqdm(jobs, desc="HP random search"):
    hp = sample_hp(trial)
    start = time.time()
    val_auc, artifact_auc = run_trial(hp, seed)
    elapsed = time.time() - start
    results.append({
        "trial": trial, "seed": seed,
        "lr": hp["lr"], "dropout": hp["dropout"], "weight_decay": hp["weight_decay"],
        "val_auc_140k": round(val_auc, 4),
        "artifact_auc": round(artifact_auc, 4),
        "time": round(elapsed, 1),
    })
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    tqdm.write(f"t{trial:02d} lr={hp['lr']:.1e} drop={hp['dropout']} wd={hp['weight_decay']:.1e} "
               f"seed={seed} | artifact={artifact_auc:.3f} | {elapsed:.0f}s")

print("HP search concluído.")

## 4. Resultados — o ajuste de HP não superou os defaults

Para saber se o "melhor" HP é **sinal ou ruído**, decompomos a variância:
- **efeito do HP** = desvio das médias *entre* configs;
- **ruído de seed** = desvio *dentro* da mesma config, trocando só a seed.

Se o ruído ≥ efeito, o ranking do topo é em boa parte sorte.

**Resultado:** 0/20 configs superaram o default (0.687); o efeito do HP (~0.010) é **menor** que o ruído de seed (~0.023). Os defaults já estavam num bom regime — o ajuste fino não trouxe ganho acima da variância entre seeds. O único sinal robusto é **negativo**: *learning rate* alto (>~3e-4) degrada a AUC cross-generator (ver as piores configs). → Levamos os **defaults** (lr=1e-4) para os notebooks 03/04, não a "melhor" config selecionada no ruído.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
agg = (df.groupby("trial")
         .agg(lr=("lr", "first"), dropout=("dropout", "first"), weight_decay=("weight_decay", "first"),
              artifact_auc=("artifact_auc", "mean"), seed_std=("artifact_auc", "std"))
         .reset_index()
         .sort_values("artifact_auc", ascending=False))
agg["seed_std"] = agg["seed_std"].fillna(0.0)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("Top 10 configs de HP (média dos seeds, augmentation jpeg+noise):\n")
print(agg.head(10).to_string(index=False))

# --- decomposição de variância: o efeito do HP é sinal ou ruído? ---
hp_effect  = agg["artifact_auc"].std()   # variação ENTRE configs  -> efeito dos HP
seed_noise = agg["seed_std"].mean()      # variação trocando só a seed (mesma config) -> ruído
default    = BASELINES["jpeg+noise (HP default)"]
best       = agg.iloc[0]
n_beat     = int((agg["artifact_auc"] > default).sum())

print("\n" + "=" * 60)
print("LEITURA HONESTA DA BUSCA")
print("=" * 60)
print(f"Efeito dos HP  (std entre configs)      = {hp_effect:.4f}")
print(f"Ruído de seed  (std média intra-config) = {seed_noise:.4f}")
print(f"Configs que superam o default ({default:.3f}) : {n_beat}/{len(agg)}")
print()

if n_beat == 0 or hp_effect <= seed_noise:
    verdict = "negativo"
    print("→ RESULTADO NEGATIVO: o efeito dos HP é MENOR que o ruído de seed e")
    print("  nenhuma config superou o default — o ranking do topo é em boa parte sorte.")
    print("  Conclusão: MANTER defaults (lr=1e-4, dropout 0.3–0.5, weight_decay=0).")
    print("  Único sinal robusto (negativo): LR alto (>~3e-4) degrada a AUC cross-generator.")
else:
    verdict = "positivo"
    print(f"→ Melhor HP supera o default: lr={best['lr']:.2e} dropout={best['dropout']} "
          f"wd={best['weight_decay']:.2e} (AUC {best['artifact_auc']:.4f})")

# salva resumo COM veredito — para não vender ruído como ganho no relatório / nos 03-04
summary = {
    "verdict": verdict,
    "hp_effect_std": round(float(hp_effect), 4),
    "seed_noise_std": round(float(seed_noise), 4),
    "n_beat_default": n_beat,
    "default_auc": default,
    "top_config": {"lr": float(best["lr"]), "dropout": float(best["dropout"]),
                   "weight_decay": float(best["weight_decay"]),
                   "artifact_auc": float(best["artifact_auc"]), "seed_std": float(best["seed_std"])},
    "recommendation": "manter defaults (lr=1e-4, dropout 0.3-0.5, wd=0); evitar lr>3e-4",
}
(RESULTS_DIR / "best_hp.json").write_text(json.dumps(summary, indent=2))
print("\nResumo (com veredito) salvo em:", RESULTS_DIR / "best_hp.json")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, logx in [(axes[0], "lr", True), (axes[1], "dropout", False), (axes[2], "weight_decay", True)]:
    ax.scatter(agg[col], agg["artifact_auc"], c="steelblue", s=40)
    if logx:
        ax.set_xscale("symlog", linthresh=1e-6)
    ax.axhline(BASELINES["jpeg+noise (HP default)"], color="tomato", linestyle="--",
               label=f"default ({BASELINES['jpeg+noise (HP default)']:.3f})")
    ax.set_xlabel(col); ax.set_ylabel("AUC ArtiFact")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle("Efeito marginal dos hiperparâmetros (augmentation jpeg+noise)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "hp_search_results.png", dpi=150, bbox_inches="tight")
plt.show()

n_beat = (agg["artifact_auc"] > BASELINES["jpeg+noise (HP default)"]).sum()
print(f"Configs que batem o default (0.687): {n_beat}/{len(agg)}")